# TP14 - Pr�dire le Nutri-Score

**Objectif :** comparer des classifieurs A-E sans fuite de donn�es, puis expliquer les erreurs observ�es.

Ce notebook pr�pare le protocole et les contr�les. Les cellules **� FAIRE** te laissent les entra�nements, choix, mesures et analyses qui constituent le cour du TP. Consigne les r�sultats dans le tableau comparatif associ�.

Donn�es : Open Food Facts contributors - licence ODbL 1.0.

## Protocole

- Conserver le split fourni par le TP13 ; ne pas le recalculer.
- Ajuster imputation, standardisation, encodage et mod�le dans chaque pli de train.
- Exclure nutriscore_score, code, product_name, brands, split et la cible des features.
- R�utiliser les m�mes plis stratifi�s et les m�mes m�triques pour chaque mod�le.
- Garder le test � l'�cart jusqu'au choix du mod�le ; l'�valuer une seule fois.
- �crire les r�sultats avant leur interpr�tation.

In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, make_scorer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file()), Path.cwd())
print(f'Racine : {ROOT.resolve()}')

## 1. Charger et contr�ler les donn�es

Le d�p�t peut contenir l'�chantillon local ou un export dat� du TP13. Si n�cessaire, d�finis NUTRISCOPE_DATA avec le chemin du Parquet.

In [ ]:
candidats = [
    Path(os.environ['NUTRISCOPE_DATA']).expanduser() if os.environ.get('NUTRISCOPE_DATA') else None,
    ROOT / 'data' / 'nutriscope_ml_sample.parquet',
    ROOT / 'data' / 'tp13' / 'exports' / 'features_nutriscore_20261125.parquet',
    ROOT / 'data-nutriscope' / 'ml' / 'nutriscope_ml_sample.parquet',
]
DATA_PATH = next((p for p in candidats if p is not None and p.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Parquet ML introuvable. D�finis NUTRISCOPE_DATA.')
df = pd.read_parquet(DATA_PATH)
print(f'{DATA_PATH.resolve()} - {len(df):,} lignes � {len(df.columns)} colonnes')
display(df.head())

In [ ]:
TARGET, SPLIT = 'nutriscore_grade', 'split'
NEVER_FEATURES = {'nutriscore_score', 'code', 'product_name', 'brands', SPLIT, TARGET}
NUMERIC_FEATURES = [
    'energy-kcal_100g', 'fat_100g', 'saturated-fat_100g', 'carbohydrates_100g',
    'sugars_100g', 'proteins_100g', 'salt_100g', 'fiber_100g',
    'fruits-vegetables-legumes_100g', 'nova_group', 'additives_n',
]
CATEGORICAL_FEATURES = ['pnns_groups_1', 'pnns_groups_2']
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
missing = {TARGET, SPLIT, *FEATURES}.difference(df.columns)
assert not missing, f'Colonnes absentes : {sorted(missing)}'
assert set(df[TARGET].dropna().unique()) <= set('abcde'), 'Grade inattendu'
assert set(df[SPLIT].dropna().unique()) == {'train', 'test'}, 'Split train/test attendu'
display(df.groupby([SPLIT, TARGET], observed=True).size().unstack(fill_value=0))
display(df[FEATURES].isna().mean().sort_values(ascending=False).rename('part_manquante'))

### Lecture des donn�es - � FAIRE

Note le volume r�ellement lu, la distribution des grades et les variables les plus incompl�tes. V�rifie que le fichier correspond au split du TP13 ; les nombres de l'�nonc� sont des rep�res, pas des r�sultats � recopier.

In [ ]:
train_df = df.loc[df[SPLIT].eq('train')].copy()
# Le test reste dans df et n'est pas transform� avant le choix final.
X_train = train_df[FEATURES].copy()
y_train = train_df[TARGET].copy()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
folds = list(cv.split(X_train, y_train))  # m�mes indices pour chaque mod�le
SCORING = {
    'f1_macro': make_scorer(f1_score, average='macro'),
    'accuracy': make_scorer(accuracy_score),
    'balanced_accuracy': make_scorer(balanced_accuracy_score),
}
print(f'Train : {len(X_train):,} lignes ; plis CV : {len(folds)}')

## 2. Baselines - � FAIRE

�value en validation crois�e la classe majoritaire, le Dummy stratifi� et des arbres de d�cision de profondeurs 3 et 8. Utilise cross_validate avec X_train, y_train, cv=folds et SCORING. Consigne moyenne et �cart-type du F1 macro, accuracy, balanced accuracy et temps dans le tableau.

**� commenter :** pourquoi l'accuracy de la baseline majoritaire para�t-elle correcte alors que le F1 macro est faible ? L'arbre peu profond pr�dit-il toutes les classes ?

In [ ]:
# � FAIRE : construire et comparer les quatre estimateurs de baseline.
# Conserver fit_time et score_time retourn�s par cross_validate.
raise NotImplementedError('Entra�ner et consigner les baselines TP14')

## 3. Pipeline et mod�les - � FAIRE

Construis un ColumnTransformer puis un Pipeline complet pour chaque mod�le : imputation des variables optionnelles avec indicateur de manque, standardisation num�rique, one-hot des cat�gories avec regroupement des modalit�s rares, puis classifieur. Les ratios ou �carts �nerg�tiques doivent �tre cr��s dans le pipeline avec FunctionTransformer. Entra�ne la r�gression logistique et la for�t al�atoire sur les m�mes plis.

�cris le contr�le qui prouve que le pr�processeur ajust� sur train n'a pas appris les statistiques du test.

In [ ]:
# � FAIRE : construire la pr�paration et les deux pipelines complets.
# �valuer avec les plis et m�triques d�j� d�finis.
raise NotImplementedError('Construire, �valuer et contr�ler les pipelines')

## 4. Comparaison et d�s�quilibre - � FAIRE

Rassemble tous les r�sultats dans un seul tableau. Ajoute la matrice de confusion du meilleur mod�le CV, normalis�e par ligne. Mesure le rapport majoritaire/minoritaire et compare class_weight='balanced'. Si tu essaies SMOTE, place-le � l'int�rieur du pipeline.

Explique l'�cart entre accuracy et F1 macro ; indique si le traitement du d�s�quilibre apporte un gain pertinent au regard de la variabilit� entre plis.

## 5. Fuite et importances - � FAIRE

Reproduis volontairement la fuite en ajoutant nutriscore_score. Mesure son F1 macro et son importance par permutation sur une validation interne au train. Explique pourquoi ce score est � l'origine du grade. Retire-le et �cris un test anti-fuite, avec un contr�le n�gatif sur une feature l�gitime.

Sur le mod�le honn�te, calcule les importances par permutation hors entra�nement et confronte-les aux composantes officielles : �nergie, sucres, acides gras satur�s, sel, fibres, prot�ines et fruits/l�gumes/l�gumineuses. Distingue fuite et proxy m�tier.

In [ ]:
# � FAIRE : fuite volontaire, importances sur validation interne, contr�les et interpr�tation.
raise NotImplementedError('Mener l'analyse de fuite et d'importance')

## 6. Test final et trois erreurs - � FAIRE

Verrouille les choix sur la CV, ajuste le mod�le retenu sur tout le train, puis construis X_test et consulte le test une seule fois. Inscris les m�triques avant leur interpr�tation.

Choisis trois erreurs r�elles de natures diff�rentes. Pour chaque produit : nutriments, explication de la pr�diction, diagnostic (apprentissage, fiche erron�e, r�gle absente des features ou r�f�rentiel diff�rent) et action. Au moins un diagnostic doit d�passer � le mod�le doit mieux apprendre �.

In [ ]:
# � FAIRE : dernier fit sur tout le train puis �valuation unique sur test.
# Conserver code, nom et nutriments pour analyser les erreurs � la main.
raise NotImplementedError('�valuer le mod�le retenu et analyser trois erreurs')

## Conclusion - � FAIRE

R�sume le meilleur mod�le et ses chiffres CV/test, la variabilit�, le pi�ge de l'accuracy, la fuite d�tect�e/corrig�e et les trois diagnostics d'erreurs. V�rifie l'�cart CV/test selon l'�nonc�. Apr�s les mesures r�elles, reporte la phrase de synth�se dans docs/journal.md.